<a href="https://colab.research.google.com/github/lidia-notebook/rag-mini-project-animalscience-IDN-/blob/main/RAG_in_AnimalSci.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Chatbot RAG (Retrieval Augmented Generation) - Animal Science

## Background

Animal Science atau Ilmu Peternakan adalah cabang ilmu yang mempelajari berbagai aspek tentang hewan ternak, mulai dari anatomi dan fisiologi, nutrisi pakan, reproduksi, genetika dan pemuliaan, kesehatan hewan, hingga manajemen produksi peternakan. Bidang ini memiliki peran penting dalam mendukung ketahanan pangan, khususnya di negara agraris seperti Indonesia, di mana sektor peternakan menjadi penyumbang utama protein hewani melalui produk daging, susu, dan telur.

Dalam praktiknya, informasi terkait Animal Science seringkali tersebar dalam berbagai literatur teknis seperti jurnal akademik, buku teks, laporan FAO, dan dokumen kementerian pertanian. Hal ini membuat pencarian informasi spesifik (misalnya kebutuhan nutrisi sapi perah, masa kebuntingan kambing, atau gejala penyakit menular) menjadi memakan waktu, terutama bagi peternak, mahasiswa, dan praktisi yang membutuhkan jawaban cepat dan akurat.

Retrieval Augmented Generation (RAG) menawarkan solusi dengan menggabungkan kekuatan Large Language Model (LLM) dan sistem pencarian dokumen berbasis embedding. Pendekatan ini memungkinkan chatbot menjawab pertanyaan teknis dengan informasi yang bersumber langsung dari knowledge base yang terstruktur, sehingga jawaban lebih akurat, dapat diverifikasi, dan tidak bergantung sepenuhnya pada pengetahuan pre-trained dari LLM yang dapat saja keliru atau usang.

Notebook ini membangun aplikasi chatbot RAG end-to-end dengan topik Animal Science, mencakup penyusunan knowledge base yang dapat diperbarui, pemilihan komponen teknis (LLM, embedding, vector database), implementasi, hingga evaluasi sistem.

---

## Daftar Isi
1. Penentuan Topik dan Ruang Lingkup Knowledge Base
2. Penyusunan Knowledge Base
3. Desain Sistem RAG
4. Implementasi Aplikasi RAG
5. Evaluasi dan Analisis Sistem
6. Insight dan Recommendation Action
7. Reflection Question


## 1. Penentuan Topik dan Ruang Lingkup Knowledge Base

**Topik yang dipilih:** Animal Science (Ilmu Peternakan dan Hewan)

### Ruang Lingkup Informasi
Knowledge base yang disusun mencakup sub-bidang utama Animal Science berikut:

1. **Anatomi dan Fisiologi Hewan Ternak** - sistem pencernaan ruminansia (sapi, kambing, domba) vs non-ruminansia (ayam, babi), sistem reproduksi, kardiovaskular, dan termoregulasi.
2. **Nutrisi dan Pakan Ternak** - kebutuhan nutrien (protein, energi, mineral, vitamin), bahan pakan, formulasi ransum, fermentasi rumen.
3. **Reproduksi Hewan** - siklus estrus, inseminasi buatan (IB), kebuntingan, manajemen pejantan dan betina.
4. **Genetika dan Pemuliaan Ternak** - seleksi, heritabilitas, persilangan, pemuliaan berbasis genom.
5. **Kesehatan Hewan dan Penyakit** - penyakit menular (PMK, antraks, ND, AI), vaksinasi, biosekuriti, kesejahteraan hewan (animal welfare).
6. **Manajemen Produksi** - sapi perah, sapi potong, ayam broiler dan petelur, kambing/domba, serta sistem perkandangan.

### Alasan Pemilihan Topik
- **Relevansi praktis**: Indonesia adalah negara agraris dengan sektor peternakan sebagai bagian penting ketahanan pangan.
- **Sumber referensi kaya**: Tersedia dari literatur akademik (FAO, jurnal peternakan), Wikipedia, dan textbook seperti Animal Science karya Taylor & Field.
- **Cocok untuk RAG**: Banyak fakta spesifik (angka kebutuhan nutrisi, masa kebuntingan, suhu tubuh normal, dll.) yang ideal untuk diambil dari dokumen daripada bergantung pada pengetahuan umum LLM yang bisa keliru.


## 2. Penyusunan Knowledge Base

### 2.1 Instalasi Library

Library yang digunakan:
- `langchain`, `langchain-community`, `langchain-groq` untuk kerangka kerja RAG
- `sentence-transformers` untuk model embedding lokal (gratis, multilingual)
- `chromadb` untuk vector database lokal
- `groq` untuk akses LLM via Groq API (tersedia free tier)


In [45]:
!pip install -q langchain langchain-community langchain-groq
!pip install -q sentence-transformers chromadb
!pip install -q groq tiktoken
print("Semua library berhasil diinstal")

Semua library berhasil diinstal


In [39]:
!pip install -q langchain langchain-community langchain-groq langchain-text-splitters
!pip install -q langchain-huggingface langchain-chroma
!pip install -q sentence-transformers chromadb
!pip install -q groq tiktoken
print("Semua library berhasil diinstal")

Semua library berhasil diinstal


### 2.2 Konfigurasi API Key (Groq)

Groq menyediakan free tier untuk LLM seperti `llama-3.3-70b-versatile`. Daftar di https://console.groq.com untuk mendapatkan API key.

Di Google Colab, simpan API key di Secrets (ikon kunci di sidebar kiri) dengan nama `GROQ_API_KEY` agar lebih aman.


In [15]:
import os

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("GROQ_API_KEY berhasil dimuat dari Colab Secrets")
except Exception:
    from getpass import getpass
    os.environ["GROQ_API_KEY"] = getpass("Masukkan GROQ_API_KEY Anda: ")
    print("GROQ_API_KEY berhasil dimasukkan")

GROQ_API_KEY berhasil dimuat dari Colab Secrets


### 2.3 Membuat Knowledge Base Awal

Knowledge base disimpan dalam file teks (`.txt`) sehingga mudah dibuka, diperbarui, atau diganti. Setiap dokumen dipisahkan oleh delimiter `===DOC===` agar nantinya bisa di-chunk per topik.


In [16]:
#  folder untuk menyimpan knowledge base
os.makedirs("knowledge_base", exist_ok=True)

# basic knowledge ttg animal science
initial_kb = """===DOC===
Judul: Sistem Pencernaan Ruminansia
Hewan ruminansia seperti sapi, kambing, domba, dan kerbau memiliki sistem pencernaan
yang unik dengan empat kompartemen lambung yaitu rumen, retikulum, omasum, dan abomasum.
Rumen adalah kompartemen terbesar dengan volume sekitar 150-200 liter pada sapi dewasa
dan berfungsi sebagai tempat fermentasi mikroba. Mikroba rumen menghasilkan Volatile
Fatty Acids (VFA) seperti asetat, propionat, dan butirat yang menjadi sumber energi
utama bagi ternak ruminansia. Proses memamah biak (rumination) memungkinkan hewan
mencerna serat kasar yang tidak bisa dicerna oleh hewan non-ruminansia.

===DOC===
Judul: Nutrisi Pakan Sapi Perah
Sapi perah membutuhkan nutrisi seimbang untuk produksi susu yang optimal. Kebutuhan
Protein Kasar (PK) sapi perah laktasi adalah 16-18% dari Bahan Kering (BK) ransum.
Total Digestible Nutrients (TDN) yang diperlukan sekitar 65-70%. Konsumsi BK harian
sapi perah berkisar 3-4% dari bobot badan. Pakan diberikan dalam bentuk hijauan
(rumput gajah, rumput odot, leguminosa) sekitar 60-70% dan konsentrat 30-40%.
Air bersih harus selalu tersedia karena sapi perah dapat minum 60-100 liter air per
hari. Mineral penting meliputi kalsium (Ca) 0.6-0.8%, fosfor (P) 0.4%, dan garam
NaCl 0.5% dari BK ransum.

===DOC===
Judul: Reproduksi dan Inseminasi Buatan Sapi
Siklus estrus (birahi) sapi rata-rata 21 hari dengan lama estrus 12-18 jam. Tanda-tanda
estrus meliputi gelisah, nafsu makan menurun, vulva bengkak dan kemerahan, keluar
lendir bening, serta mau dinaiki sapi lain (standing heat). Waktu terbaik untuk
Inseminasi Buatan (IB) adalah 10-14 jam setelah tanda estrus pertama terlihat,
mengikuti aturan AM-PM rule. Masa kebuntingan (gestasi) sapi adalah 280-285 hari
atau sekitar 9 bulan 10 hari. Sapi dara biasanya dikawinkan pertama kali pada umur
15-18 bulan dengan bobot minimal 280-300 kg.

===DOC===
Judul: Genetika dan Pemuliaan Ternak
Pemuliaan ternak (animal breeding) bertujuan meningkatkan mutu genetik ternak melalui
seleksi dan persilangan. Heritabilitas (h kuadrat) adalah proporsi ragam genetik aditif
terhadap ragam fenotipik total, dengan nilai 0-1. Sifat dengan h kuadrat tinggi (di atas 0.4)
seperti tinggi badan dan bobot dewasa lebih responsif terhadap seleksi. Sifat dengan
h kuadrat rendah (di bawah 0.2) seperti jumlah anak per kelahiran lebih dipengaruhi lingkungan.
Persilangan (crossbreeding) menghasilkan heterosis atau vigor hibrida, contohnya sapi
PO (Peranakan Ongole) hasil persilangan sapi Ongole dengan sapi lokal. Estimated
Breeding Value (EBV) digunakan untuk memprediksi nilai pemuliaan ternak.

===DOC===
Judul: Penyakit Mulut dan Kuku (PMK)
Penyakit Mulut dan Kuku (PMK) atau Foot and Mouth Disease (FMD) adalah penyakit
viral sangat menular yang menyerang hewan berkuku belah seperti sapi, kerbau, babi,
kambing, dan domba. Penyebabnya adalah virus Aphthovirus famili Picornaviridae
dengan 7 serotipe (O, A, C, SAT1, SAT2, SAT3, Asia1). Gejala klinis meliputi demam
tinggi (40-41 derajat C), lepuh (vesikel) pada mulut, lidah, gusi, dan sela kuku, hipersalivasi,
serta pincang. Tingkat morbiditas dapat mencapai 100% tetapi mortalitas rendah (1-5%)
kecuali pada hewan muda. Pencegahan dilakukan dengan vaksinasi rutin setiap 6 bulan,
biosekuriti ketat, dan pembatasan lalu lintas ternak.

===DOC===
Judul: Manajemen Ayam Broiler
Ayam broiler adalah ayam pedaging yang dipelihara untuk produksi daging dengan
masa panen 28-35 hari pada bobot 1.5-2.2 kg. Suhu kandang berbeda menurut umur:
DOC (Day Old Chick) 32-34 derajat C, minggu 2 sekitar 29-30 derajat C, minggu 3 sekitar 27-28 derajat C,
minggu 4 ke atas 24-26 derajat C. Feed Conversion Ratio (FCR) yang baik adalah 1.5-1.7,
artinya 1 kg pakan menghasilkan kenaikan bobot 0.6-0.7 kg. Mortalitas normal di
bawah 5%. Kepadatan kandang 8-10 ekor per m persegi untuk iklim tropis. Vaksinasi rutin
meliputi ND (Newcastle Disease), Gumboro (IBD), dan AI (Avian Influenza).

===DOC===
Judul: Sapi Potong dan Penggemukan
Sapi potong dipelihara untuk produksi daging dengan target Pertambahan Bobot Badan
Harian (PBBH) 0.8-1.2 kg/hari pada sistem feedlot intensif. Bangsa sapi potong populer
di Indonesia antara lain Limousin, Simmental, Brahman, dan sapi lokal seperti Bali
dan Madura. Lama penggemukan 90-120 hari dengan bobot awal 250-350 kg dan bobot
akhir 450-550 kg. Rasio pakan konsentrat:hijauan untuk penggemukan adalah 70:30
hingga 80:20. Karkas yang baik memiliki persentase karkas 50-55% dari bobot hidup.
Marbling atau perlemakan intramuskular menentukan kualitas daging premium.

===DOC===
Judul: Kambing dan Domba
Kambing dan domba termasuk ruminansia kecil yang efisien mengkonversi hijauan menjadi
daging dan susu. Masa kebuntingan kambing/domba adalah 145-155 hari (sekitar 5 bulan).
Siklus estrus 17-21 hari dengan lama estrus 24-36 jam. Bangsa kambing populer di
Indonesia: Etawa (PE), Kacang, Boer, dan Saanen (perah). Bangsa domba: Domba Garut,
Ekor Tipis, dan Merino. Litter size (jumlah anak per kelahiran) 1-3 ekor dengan
rata-rata 1.5-2. Bobot dewasa kambing PE jantan 60-90 kg, betina 40-60 kg. Produksi
susu kambing Saanen mencapai 2-3 liter/hari.

===DOC===
Judul: Kesejahteraan Hewan (Animal Welfare)
Kesejahteraan hewan mengacu pada lima kebebasan (Five Freedoms): (1) bebas dari rasa
lapar dan haus, (2) bebas dari rasa tidak nyaman, (3) bebas dari rasa sakit, luka,
dan penyakit, (4) bebas mengekspresikan perilaku alami, (5) bebas dari rasa takut
dan tertekan. Indikator kesejahteraan meliputi kondisi tubuh (Body Condition Score),
perilaku, kesehatan, dan produktivitas. Praktik animal welfare yang baik mencakup
kandang yang sesuai, pakan dan air cukup, penanganan yang halus saat handling,
serta pemotongan yang humanis (humane slaughter) dengan metode stunning atau
penyembelihan sesuai syariat untuk hewan halal.
"""

with open("knowledge_base/animal_science.txt", "w", encoding="utf-8") as f:
    f.write(initial_kb)

print("Knowledge base awal berhasil dibuat")
print("Lokasi file: knowledge_base/animal_science.txt")
print(f"Jumlah dokumen: {initial_kb.count('===DOC===')}")

Knowledge base awal berhasil dibuat
Lokasi file: knowledge_base/animal_science.txt
Jumlah dokumen: 9


### 2.4 Fungsi untuk Memperbarui Knowledge Base

Knowledge base dirancang dapat diperbarui dengan dua mekanisme:
1. `add_document()` untuk menambahkan dokumen baru
2. `replace_knowledge_base()` untuk mengganti seluruh isi knowledge base


In [17]:
def add_document(title: str, content: str, kb_path: str = "knowledge_base/animal_science.txt"):
    """Menambahkan dokumen baru ke knowledge base."""
    new_doc = f"\n===DOC===\nJudul: {title}\n{content.strip()}\n"
    with open(kb_path, "a", encoding="utf-8") as f:
        f.write(new_doc)
    print(f"Dokumen '{title}' berhasil ditambahkan ke knowledge base")


def replace_knowledge_base(new_content: str, kb_path: str = "knowledge_base/animal_science.txt"):
    """Mengganti seluruh isi knowledge base."""
    with open(kb_path, "w", encoding="utf-8") as f:
        f.write(new_content)
    print("Knowledge base berhasil diganti seluruhnya")


def view_knowledge_base(kb_path: str = "knowledge_base/animal_science.txt"):
    """Menampilkan isi knowledge base."""
    with open(kb_path, "r", encoding="utf-8") as f:
        content = f.read()
    print(f"Total dokumen: {content.count('===DOC===')}")
    print(f"Total karakter: {len(content)}")
    return content


content = view_knowledge_base()
print("\n--- Preview 500 karakter pertama ---")
print(content[:500])

Total dokumen: 9
Total karakter: 5811

--- Preview 500 karakter pertama ---
===DOC===
Judul: Sistem Pencernaan Ruminansia
Hewan ruminansia seperti sapi, kambing, domba, dan kerbau memiliki sistem pencernaan
yang unik dengan empat kompartemen lambung yaitu rumen, retikulum, omasum, dan abomasum.
Rumen adalah kompartemen terbesar dengan volume sekitar 150-200 liter pada sapi dewasa
dan berfungsi sebagai tempat fermentasi mikroba. Mikroba rumen menghasilkan Volatile
Fatty Acids (VFA) seperti asetat, propionat, dan butirat yang menjadi sumber energi
utama bagi ternak rumina


## 3. Desain Sistem RAG

### 3.1 Komponen Utama Sistem

| Komponen | Pilihan | Alasan |
|---------|---------|--------|
| **LLM** | `llama-3.3-70b-versatile` (via Groq API) | Free tier, inferensi sangat cepat (LPU), mendukung bahasa Indonesia dengan baik, ukuran 70B parameter cukup besar untuk reasoning yang baik. |
| **Embedding Model** | `paraphrase-multilingual-MiniLM-L12-v2` (Sentence Transformers) | Mendukung bahasa Indonesia dan Inggris, ukuran kecil (sekitar 120MB) sehingga cepat di-load di Colab, gratis, tidak butuh API. |
| **Vector Database** | **ChromaDB** (local persistent) | Open-source, ringan, mudah di-setup di Colab, mendukung persistensi data ke disk sehingga embedding tidak perlu di-recompute setiap kali. |

### 3.2 Alasan Pemilihan Komponen secara Detail

**LLM - Llama 3.3 70B via Groq:**
- Groq menggunakan LPU (Language Processing Unit) yang jauh lebih cepat dari GPU konvensional, sehingga response time kurang dari 1 detik.
- Free tier mencukupi untuk eksperimen (ribuan request per hari).
- Llama 3.3 70B unggul dalam instruction following dan menjawab dalam bahasa Indonesia secara natural.
- Karakteristik topik Animal Science memerlukan reasoning lintas konsep (misalnya menghubungkan nutrisi dengan produksi susu), sehingga butuh model dengan kapasitas reasoning yang baik.

**Embedding - Multilingual MiniLM:**
- Knowledge base ditulis dalam bahasa Indonesia, sehingga embedding monolingual (English-only) akan menghasilkan kualitas retrieval buruk.
- MiniLM-L12-v2 versi multilingual sudah dilatih pada 50+ bahasa termasuk Indonesia.
- Dimensi vector 384 (kecil) sehingga cepat untuk similarity search di Colab.

**Vector DB - ChromaDB:**
- Tidak perlu setup server terpisah (seperti Pinecone atau Weaviate).
- Mendukung metadata filtering yang bermanfaat jika kelak ingin memfilter dokumen berdasarkan kategori (misal hanya Reproduksi atau hanya Nutrisi).
- Persistence ke disk memungkinkan KB tetap ada meski session Colab restart (asalkan disimpan ke Google Drive).


## 4. Implementasi Aplikasi RAG

### 4.1 Loading dan Chunking Dokumen

Setiap dokumen dipecah menjadi chunks yang lebih kecil agar retrieval lebih presisi. Digunakan `RecursiveCharacterTextSplitter` dengan `chunk_size=400` dan `chunk_overlap=50`.


In [18]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


def load_and_chunk_kb(kb_path: str = "knowledge_base/animal_science.txt"):
    """Memuat knowledge base dan memecahnya per dokumen, lalu di-chunk."""
    with open(kb_path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    # memisahkan per dokumen berdasarkan delimiter
    raw_docs = [d.strip() for d in raw_text.split("===DOC===") if d.strip()]

    # bungkus dalam objek Document dengan metadata judul
    documents = []
    for doc in raw_docs:
        lines = doc.split("\n", 1)
        title = lines[0].replace("Judul:", "").strip() if lines[0].startswith("Judul:") else "Untitled"
        content = lines[1] if len(lines) > 1 else doc
        documents.append(Document(page_content=content, metadata={"title": title}))

    # Chunking
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=50,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_documents(documents)
    return chunks


chunks = load_and_chunk_kb()
print(f"Total dokumen: {len(set(c.metadata['title'] for c in chunks))}")
print(f"Total chunks: {len(chunks)}")
print("\n--- Contoh chunk pertama ---")
print(f"Title: {chunks[0].metadata['title']}")
print(f"Content: {chunks[0].page_content[:200]}...")

Total dokumen: 9
Total chunks: 18

--- Contoh chunk pertama ---
Title: Sistem Pencernaan Ruminansia
Content: Hewan ruminansia seperti sapi, kambing, domba, dan kerbau memiliki sistem pencernaan
yang unik dengan empat kompartemen lambung yaitu rumen, retikulum, omasum, dan abomasum.
Rumen adalah kompartemen t...


### 4.2 Membuat Embedding dan Menyimpan ke ChromaDB

In [41]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Inisialisasi model embedding multilingual
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
print("Model embedding berhasil dimuat")


def build_vectorstore(chunks):
    """Bangun vector store IN-MEMORY (tanpa persist ke disk)."""
    new_vs = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        collection_name="animal_science_kb"
    )
    return new_vs


vectorstore = build_vectorstore(chunks)
print("Vector store berhasil dibuat (in-memory)")
print(f"Jumlah vector: {vectorstore._collection.count()}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model embedding berhasil dimuat
Vector store berhasil dibuat (in-memory)
Jumlah vector: 18


### 4.3 Fungsi untuk Rebuild Vector Store setelah KB di-update

Setiap kali knowledge base diubah (tambah atau ganti), vector store harus di-rebuild agar perubahan tercermin dalam retrieval.


In [20]:
def rebuild_vectorstore(kb_path: str = "knowledge_base/animal_science.txt"):
    """Rebuild vector store setelah knowledge base di-update."""
    print("Loading dan chunking knowledge base baru...")
    new_chunks = load_and_chunk_kb(kb_path)
    print(f"Total chunks baru: {len(new_chunks)}")

    print("Building vector store baru...")
    new_vs = build_vectorstore(new_chunks)
    print("Vector store berhasil di-rebuild")
    print(f"Jumlah vector: {new_vs._collection.count()}")
    return new_vs

### 4.4 Membangun Chain RAG dengan Prompt Template

In [23]:
!pip install -q langchain langchain-core langchain-community
print("Installation done. Sekarang restart runtime: Runtime → Restart session")

Installation done. Sekarang restart runtime: Runtime → Restart session


In [25]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Inisialisasi LLM Groq
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,  # rendah agar jawaban lebih faktual
    max_tokens=1024
)

# Prompt template kustom dalam bahasa Indonesia
prompt_template = """Anda adalah asisten ahli di bidang Animal Science (Ilmu Peternakan).
Jawablah pertanyaan pengguna HANYA berdasarkan KONTEKS yang diberikan di bawah ini.

Aturan menjawab:
1. Jika informasi tidak ada dalam konteks, katakan "Maaf, informasi tersebut tidak tersedia dalam knowledge base saya."
2. Jawab dalam bahasa Indonesia yang jelas dan ringkas.
3. Sertakan angka, fakta, dan detail spesifik dari konteks jika relevan.
4. Jangan mengarang informasi.

KONTEKS:
{context}

PERTANYAAN: {question}

JAWABAN:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)


def format_docs(docs):
    """Gabungkan isi dokumen menjadi satu string konteks."""
    return "\n\n".join(doc.page_content for doc in docs)


def build_rag_chain(vs):
    """Membangun RAG chain dengan retriever dari vector store (LCEL style)."""
    retriever = vs.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3}  # ambil top 3
    )

    # LCEL chain: input -> retrieve docs + passthrough question -> format -> prompt -> LLM -> string
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | PROMPT
        | llm
        | StrOutputParser()
    )


    return {"chain": rag_chain, "retriever": retriever}


qa_chain = build_rag_chain(vectorstore)
print("RAG chain berhasil dibangun")

RAG chain berhasil dibangun


### 4.5 Fungsi Chatbot untuk Bertanya

In [28]:
def ask(question: str, chain=None):
    """Mengajukan pertanyaan ke chatbot RAG."""
    if chain is None:
        chain = qa_chain

    print(f"Pertanyaan: {question}\n")

    # ambil answer dari chain (LCEL style)
    answer = chain["chain"].invoke(question)
    # ambil source documents dari retriever
    source_docs = chain["retriever"].invoke(question)

    print(f"Jawaban:\n{answer}\n")

    print("Sumber yang digunakan:")
    for i, doc in enumerate(source_docs, 1):
        title = doc.metadata.get("title", "Untitled")
        snippet = doc.page_content[:120].replace("\n", " ")
        print(f"  [{i}] {title} - \"{snippet}...\"")
    print("-" * 70)
    return {"result": answer, "source_documents": source_docs}

## 5. Evaluasi dan Analisis Sistem

### 5.1 Pengujian dengan Beberapa Pertanyaan

Kita akan menguji chatbot dengan berbagai kategori pertanyaan:
1. Pertanyaan faktual spesifik (angka, durasi)
2. Pertanyaan konseptual (definisi, mekanisme)
3. Pertanyaan perbandingan atau multi-dokumen
4. Pertanyaan di luar knowledge base (untuk uji hallucination)


In [30]:
# Test 1: Pertanyaan spesifik (yang ada range aslinya)
ask("Berapa lama masa kebuntingan sapi?")

Pertanyaan: Berapa lama masa kebuntingan sapi?

Jawaban:
Masa kebuntingan sapi adalah 280-285 hari atau sekitar 9 bulan 10 hari.

Sumber yang digunakan:
  [1] Reproduksi dan Inseminasi Buatan Sapi - "mengikuti aturan AM-PM rule. Masa kebuntingan (gestasi) sapi adalah 280-285 hari atau sekitar 9 bulan 10 hari. Sapi dara..."
  [2] Kambing dan Domba - "Kambing dan domba termasuk ruminansia kecil yang efisien mengkonversi hijauan menjadi daging dan susu. Masa kebuntingan ..."
  [3] Reproduksi dan Inseminasi Buatan Sapi - "Siklus estrus (birahi) sapi rata-rata 21 hari dengan lama estrus 12-18 jam. Tanda-tanda estrus meliputi gelisah, nafsu m..."
----------------------------------------------------------------------


{'result': 'Masa kebuntingan sapi adalah 280-285 hari atau sekitar 9 bulan 10 hari.',
 'source_documents': [Document(metadata={'title': 'Reproduksi dan Inseminasi Buatan Sapi'}, page_content='mengikuti aturan AM-PM rule. Masa kebuntingan (gestasi) sapi adalah 280-285 hari\natau sekitar 9 bulan 10 hari. Sapi dara biasanya dikawinkan pertama kali pada umur\n15-18 bulan dengan bobot minimal 280-300 kg.'),
  Document(metadata={'title': 'Kambing dan Domba'}, page_content='Kambing dan domba termasuk ruminansia kecil yang efisien mengkonversi hijauan menjadi\ndaging dan susu. Masa kebuntingan kambing/domba adalah 145-155 hari (sekitar 5 bulan).\nSiklus estrus 17-21 hari dengan lama estrus 24-36 jam. Bangsa kambing populer di\nIndonesia: Etawa (PE), Kacang, Boer, dan Saanen (perah). Bangsa domba: Domba Garut,'),
  Document(metadata={'title': 'Reproduksi dan Inseminasi Buatan Sapi'}, page_content='Siklus estrus (birahi) sapi rata-rata 21 hari dengan lama estrus 12-18 jam. Tanda-tanda\nestrus me

In [31]:
# Test 2: Pertanyaan konseptual
ask("Apa yang dimaksud dengan heritabilitas dalam pemuliaan ternak?")

Pertanyaan: Apa yang dimaksud dengan heritabilitas dalam pemuliaan ternak?

Jawaban:
Heritabilitas (h kuadrat) adalah proporsi ragam genetik aditif terhadap ragam fenotipik total, dengan nilai 0-1, yang menunjukkan seberapa besar sifat tertentu dapat dipengaruhi oleh faktor genetik.

Sumber yang digunakan:
  [1] Genetika dan Pemuliaan Ternak - "Pemuliaan ternak (animal breeding) bertujuan meningkatkan mutu genetik ternak melalui seleksi dan persilangan. Heritabil..."
  [2] Kesejahteraan Hewan (Animal Welfare) - "perilaku, kesehatan, dan produktivitas. Praktik animal welfare yang baik mencakup kandang yang sesuai, pakan dan air cuk..."
  [3] Kesejahteraan Hewan (Animal Welfare) - "Kesejahteraan hewan mengacu pada lima kebebasan (Five Freedoms): (1) bebas dari rasa lapar dan haus, (2) bebas dari rasa..."
----------------------------------------------------------------------


{'result': 'Heritabilitas (h kuadrat) adalah proporsi ragam genetik aditif terhadap ragam fenotipik total, dengan nilai 0-1, yang menunjukkan seberapa besar sifat tertentu dapat dipengaruhi oleh faktor genetik.',
 'source_documents': [Document(metadata={'title': 'Genetika dan Pemuliaan Ternak'}, page_content='Pemuliaan ternak (animal breeding) bertujuan meningkatkan mutu genetik ternak melalui\nseleksi dan persilangan. Heritabilitas (h kuadrat) adalah proporsi ragam genetik aditif\nterhadap ragam fenotipik total, dengan nilai 0-1. Sifat dengan h kuadrat tinggi (di atas 0.4)\nseperti tinggi badan dan bobot dewasa lebih responsif terhadap seleksi. Sifat dengan'),
  Document(metadata={'title': 'Kesejahteraan Hewan (Animal Welfare)'}, page_content='perilaku, kesehatan, dan produktivitas. Praktik animal welfare yang baik mencakup\nkandang yang sesuai, pakan dan air cukup, penanganan yang halus saat handling,\nserta pemotongan yang humanis (humane slaughter) dengan metode stunning atau\npeny

In [32]:
# Test 3: Pertanyaan multi-dokumen (memerlukan retrieval lintas topik)
ask("Apa saja vaksin yang perlu diberikan pada ayam broiler?")

Pertanyaan: Apa saja vaksin yang perlu diberikan pada ayam broiler?

Jawaban:
Vaksin yang perlu diberikan pada ayam broiler meliputi ND (Newcastle Disease), Gumboro (IBD), dan AI (Avian Influenza).

Sumber yang digunakan:
  [1] Manajemen Ayam Broiler - "Ayam broiler adalah ayam pedaging yang dipelihara untuk produksi daging dengan masa panen 28-35 hari pada bobot 1.5-2.2 ..."
  [2] Penyakit Mulut dan Kuku (PMK) - "tinggi (40-41 derajat C), lepuh (vesikel) pada mulut, lidah, gusi, dan sela kuku, hipersalivasi, serta pincang. Tingkat ..."
  [3] Manajemen Ayam Broiler - "artinya 1 kg pakan menghasilkan kenaikan bobot 0.6-0.7 kg. Mortalitas normal di bawah 5%. Kepadatan kandang 8-10 ekor pe..."
----------------------------------------------------------------------


{'result': 'Vaksin yang perlu diberikan pada ayam broiler meliputi ND (Newcastle Disease), Gumboro (IBD), dan AI (Avian Influenza).',
 'source_documents': [Document(metadata={'title': 'Manajemen Ayam Broiler'}, page_content='Ayam broiler adalah ayam pedaging yang dipelihara untuk produksi daging dengan\nmasa panen 28-35 hari pada bobot 1.5-2.2 kg. Suhu kandang berbeda menurut umur:\nDOC (Day Old Chick) 32-34 derajat C, minggu 2 sekitar 29-30 derajat C, minggu 3 sekitar 27-28 derajat C,\nminggu 4 ke atas 24-26 derajat C. Feed Conversion Ratio (FCR) yang baik adalah 1.5-1.7,'),
  Document(metadata={'title': 'Penyakit Mulut dan Kuku (PMK)'}, page_content='tinggi (40-41 derajat C), lepuh (vesikel) pada mulut, lidah, gusi, dan sela kuku, hipersalivasi,\nserta pincang. Tingkat morbiditas dapat mencapai 100% tetapi mortalitas rendah (1-5%)\nkecuali pada hewan muda. Pencegahan dilakukan dengan vaksinasi rutin setiap 6 bulan,\nbiosekuriti ketat, dan pembatasan lalu lintas ternak.'),
  Document(

In [33]:
# Test 4: Pertanyaan tentang PMK
ask("Apa gejala penyakit mulut dan kuku pada sapi?")

Pertanyaan: Apa gejala penyakit mulut dan kuku pada sapi?

Jawaban:
Gejala klinis penyakit mulut dan kuku (PMK) pada sapi meliputi demam tinggi (40-41 derajat C), lepuh (vesikel) pada mulut, lidah, gusi, dan sela kuku, hipersalivasi, serta pincang.

Sumber yang digunakan:
  [1] Penyakit Mulut dan Kuku (PMK) - "tinggi (40-41 derajat C), lepuh (vesikel) pada mulut, lidah, gusi, dan sela kuku, hipersalivasi, serta pincang. Tingkat ..."
  [2] Penyakit Mulut dan Kuku (PMK) - "Penyakit Mulut dan Kuku (PMK) atau Foot and Mouth Disease (FMD) adalah penyakit viral sangat menular yang menyerang hewa..."
  [3] Nutrisi Pakan Sapi Perah - "Sapi perah membutuhkan nutrisi seimbang untuk produksi susu yang optimal. Kebutuhan Protein Kasar (PK) sapi perah laktas..."
----------------------------------------------------------------------


{'result': 'Gejala klinis penyakit mulut dan kuku (PMK) pada sapi meliputi demam tinggi (40-41 derajat C), lepuh (vesikel) pada mulut, lidah, gusi, dan sela kuku, hipersalivasi, serta pincang.',
 'source_documents': [Document(metadata={'title': 'Penyakit Mulut dan Kuku (PMK)'}, page_content='tinggi (40-41 derajat C), lepuh (vesikel) pada mulut, lidah, gusi, dan sela kuku, hipersalivasi,\nserta pincang. Tingkat morbiditas dapat mencapai 100% tetapi mortalitas rendah (1-5%)\nkecuali pada hewan muda. Pencegahan dilakukan dengan vaksinasi rutin setiap 6 bulan,\nbiosekuriti ketat, dan pembatasan lalu lintas ternak.'),
  Document(metadata={'title': 'Penyakit Mulut dan Kuku (PMK)'}, page_content='Penyakit Mulut dan Kuku (PMK) atau Foot and Mouth Disease (FMD) adalah penyakit\nviral sangat menular yang menyerang hewan berkuku belah seperti sapi, kerbau, babi,\nkambing, dan domba. Penyebabnya adalah virus Aphthovirus famili Picornaviridae\ndengan 7 serotipe (O, A, C, SAT1, SAT2, SAT3, Asia1). G

In [34]:
# Test 5: Pertanyaan tentang nutrisi
ask("Berapa kebutuhan protein kasar untuk sapi perah laktasi?")

Pertanyaan: Berapa kebutuhan protein kasar untuk sapi perah laktasi?

Jawaban:
Kebutuhan Protein Kasar (PK) sapi perah laktasi adalah 16-18% dari Bahan Kering (BK) ransum.

Sumber yang digunakan:
  [1] Nutrisi Pakan Sapi Perah - "Sapi perah membutuhkan nutrisi seimbang untuk produksi susu yang optimal. Kebutuhan Protein Kasar (PK) sapi perah laktas..."
  [2] Sistem Pencernaan Ruminansia - "Hewan ruminansia seperti sapi, kambing, domba, dan kerbau memiliki sistem pencernaan yang unik dengan empat kompartemen ..."
  [3] Sapi Potong dan Penggemukan - "Sapi potong dipelihara untuk produksi daging dengan target Pertambahan Bobot Badan Harian (PBBH) 0.8-1.2 kg/hari pada si..."
----------------------------------------------------------------------


{'result': 'Kebutuhan Protein Kasar (PK) sapi perah laktasi adalah 16-18% dari Bahan Kering (BK) ransum.',
 'source_documents': [Document(metadata={'title': 'Nutrisi Pakan Sapi Perah'}, page_content='Sapi perah membutuhkan nutrisi seimbang untuk produksi susu yang optimal. Kebutuhan\nProtein Kasar (PK) sapi perah laktasi adalah 16-18% dari Bahan Kering (BK) ransum.\nTotal Digestible Nutrients (TDN) yang diperlukan sekitar 65-70%. Konsumsi BK harian\nsapi perah berkisar 3-4% dari bobot badan. Pakan diberikan dalam bentuk hijauan'),
  Document(metadata={'title': 'Sistem Pencernaan Ruminansia'}, page_content='Hewan ruminansia seperti sapi, kambing, domba, dan kerbau memiliki sistem pencernaan\nyang unik dengan empat kompartemen lambung yaitu rumen, retikulum, omasum, dan abomasum.\nRumen adalah kompartemen terbesar dengan volume sekitar 150-200 liter pada sapi dewasa\ndan berfungsi sebagai tempat fermentasi mikroba. Mikroba rumen menghasilkan Volatile'),
  Document(metadata={'title': 'Sap

In [35]:
# Test 6: Pertanyaan di luar knowledge base (uji hallucination)
ask("Bagaimana cara membuat kue brownies?")

Pertanyaan: Bagaimana cara membuat kue brownies?

Jawaban:
Maaf, informasi tersebut tidak tersedia dalam knowledge base saya.

Sumber yang digunakan:
  [1] Sapi Potong dan Penggemukan - "akhir 450-550 kg. Rasio pakan konsentrat:hijauan untuk penggemukan adalah 70:30 hingga 80:20. Karkas yang baik memiliki ..."
  [2] Sapi Potong dan Penggemukan - "Sapi potong dipelihara untuk produksi daging dengan target Pertambahan Bobot Badan Harian (PBBH) 0.8-1.2 kg/hari pada si..."
  [3] Nutrisi Pakan Sapi Perah - "Sapi perah membutuhkan nutrisi seimbang untuk produksi susu yang optimal. Kebutuhan Protein Kasar (PK) sapi perah laktas..."
----------------------------------------------------------------------


{'result': 'Maaf, informasi tersebut tidak tersedia dalam knowledge base saya.',
 'source_documents': [Document(metadata={'title': 'Sapi Potong dan Penggemukan'}, page_content='akhir 450-550 kg. Rasio pakan konsentrat:hijauan untuk penggemukan adalah 70:30\nhingga 80:20. Karkas yang baik memiliki persentase karkas 50-55% dari bobot hidup.\nMarbling atau perlemakan intramuskular menentukan kualitas daging premium.'),
  Document(metadata={'title': 'Sapi Potong dan Penggemukan'}, page_content='Sapi potong dipelihara untuk produksi daging dengan target Pertambahan Bobot Badan\nHarian (PBBH) 0.8-1.2 kg/hari pada sistem feedlot intensif. Bangsa sapi potong populer\ndi Indonesia antara lain Limousin, Simmental, Brahman, dan sapi lokal seperti Bali\ndan Madura. Lama penggemukan 90-120 hari dengan bobot awal 250-350 kg dan bobot'),
  Document(metadata={'title': 'Nutrisi Pakan Sapi Perah'}, page_content='Sapi perah membutuhkan nutrisi seimbang untuk produksi susu yang optimal. Kebutuhan\nProtein

### 5.2 Uji Coba Fitur Update Knowledge Base

Mari kita demonstrasikan kemampuan menambah dokumen baru ke knowledge base.


In [36]:
# Tambahkan dokumen baru tentang sapi Bali
add_document(
    title="Sapi Bali",
    content="""Sapi Bali (Bos javanicus) adalah sapi asli Indonesia yang merupakan hasil
domestikasi banteng. Sapi Bali memiliki ciri khas warna merah bata pada betina dan
hitam pada jantan dewasa, dengan garis hitam (eel stripe) sepanjang punggung dan
pantat berwarna putih (white mirror). Bobot dewasa jantan 350-450 kg dan betina
250-350 kg. Sapi Bali memiliki keunggulan adaptasi tropis yang sangat baik, fertilitas
tinggi (calving interval 14-16 bulan), dan persentase karkas mencapai 55-58%. Sapi Bali
banyak dikembangkan di Bali, NTB, NTT, dan Sulawesi sebagai sapi potong unggulan."""
)

# Tambahkan dokumen baru tentang pakan fermentasi
add_document(
    title="Pakan Fermentasi (Silase)",
    content="""Silase adalah pakan hijauan yang diawetkan melalui proses fermentasi anaerob
oleh bakteri asam laktat. Bahan yang umum digunakan adalah rumput gajah, jagung, atau
sorgum dengan kadar air 60-70%. Proses fermentasi berlangsung 21 hari dalam kondisi
anaerob (tanpa oksigen). pH silase yang baik adalah 3.8-4.2. Penambahan molases 3-5%
dapat mempercepat fermentasi. Keunggulan silase: dapat disimpan 6-12 bulan, mengurangi
kehilangan nutrien dibanding hijauan kering, dan tersedia sepanjang tahun terutama di
musim kemarau."""
)

print("\nKnowledge base setelah update:")
view_knowledge_base()

Dokumen 'Sapi Bali' berhasil ditambahkan ke knowledge base
Dokumen 'Pakan Fermentasi (Silase)' berhasil ditambahkan ke knowledge base

Knowledge base setelah update:
Total dokumen: 11
Total karakter: 6978


'===DOC===\nJudul: Sistem Pencernaan Ruminansia\nHewan ruminansia seperti sapi, kambing, domba, dan kerbau memiliki sistem pencernaan\nyang unik dengan empat kompartemen lambung yaitu rumen, retikulum, omasum, dan abomasum.\nRumen adalah kompartemen terbesar dengan volume sekitar 150-200 liter pada sapi dewasa\ndan berfungsi sebagai tempat fermentasi mikroba. Mikroba rumen menghasilkan Volatile\nFatty Acids (VFA) seperti asetat, propionat, dan butirat yang menjadi sumber energi\nutama bagi ternak ruminansia. Proses memamah biak (rumination) memungkinkan hewan\nmencerna serat kasar yang tidak bisa dicerna oleh hewan non-ruminansia.\n\n===DOC===\nJudul: Nutrisi Pakan Sapi Perah\nSapi perah membutuhkan nutrisi seimbang untuk produksi susu yang optimal. Kebutuhan\nProtein Kasar (PK) sapi perah laktasi adalah 16-18% dari Bahan Kering (BK) ransum.\nTotal Digestible Nutrients (TDN) yang diperlukan sekitar 65-70%. Konsumsi BK harian\nsapi perah berkisar 3-4% dari bobot badan. Pakan diberikan d

In [42]:
# Rebuild vector store agar dokumen baru ter-index
vectorstore = rebuild_vectorstore()
qa_chain = build_rag_chain(vectorstore)
print("\nChain berhasil di-rebuild dengan knowledge base terbaru")

Loading dan chunking knowledge base baru...
Total chunks baru: 22
Building vector store baru...
Vector store berhasil di-rebuild
Jumlah vector: 40

Chain berhasil di-rebuild dengan knowledge base terbaru


In [43]:
# Uji pertanyaan tentang dokumen yang baru ditambahkan
ask("Apa keunggulan sapi Bali sebagai sapi potong?")

Pertanyaan: Apa keunggulan sapi Bali sebagai sapi potong?

Jawaban:
Keunggulan sapi Bali sebagai sapi potong adalah adaptasi tropis yang sangat baik, fertilitas tinggi (calving interval 14-16 bulan), dan persentase karkas mencapai 55-58%.

Sumber yang digunakan:
  [1] Sapi Bali - "Sapi Bali (Bos javanicus) adalah sapi asli Indonesia yang merupakan hasil domestikasi banteng. Sapi Bali memiliki ciri k..."
  [2] Sapi Bali - "250-350 kg. Sapi Bali memiliki keunggulan adaptasi tropis yang sangat baik, fertilitas tinggi (calving interval 14-16 bu..."
  [3] Sapi Potong dan Penggemukan - "Sapi potong dipelihara untuk produksi daging dengan target Pertambahan Bobot Badan Harian (PBBH) 0.8-1.2 kg/hari pada si..."
----------------------------------------------------------------------


{'result': 'Keunggulan sapi Bali sebagai sapi potong adalah adaptasi tropis yang sangat baik, fertilitas tinggi (calving interval 14-16 bulan), dan persentase karkas mencapai 55-58%.',
 'source_documents': [Document(id='ebfd1b74-ddfa-400f-9b15-199605862960', metadata={'title': 'Sapi Bali'}, page_content='Sapi Bali (Bos javanicus) adalah sapi asli Indonesia yang merupakan hasil\ndomestikasi banteng. Sapi Bali memiliki ciri khas warna merah bata pada betina dan\nhitam pada jantan dewasa, dengan garis hitam (eel stripe) sepanjang punggung dan\npantat berwarna putih (white mirror). Bobot dewasa jantan 350-450 kg dan betina'),
  Document(id='52de40c1-2f26-463f-afda-19a32df7743c', metadata={'title': 'Sapi Bali'}, page_content='250-350 kg. Sapi Bali memiliki keunggulan adaptasi tropis yang sangat baik, fertilitas\ntinggi (calving interval 14-16 bulan), dan persentase karkas mencapai 55-58%. Sapi Bali\nbanyak dikembangkan di Bali, NTB, NTT, dan Sulawesi sebagai sapi potong unggulan.'),
  Docum

In [44]:
# Uji pertanyaan tentang silase
ask("Berapa lama proses fermentasi silase dan berapa pH yang ideal?")

Pertanyaan: Berapa lama proses fermentasi silase dan berapa pH yang ideal?

Jawaban:
Proses fermentasi silase berlangsung selama 21 hari, dan pH yang ideal adalah 3.8-4.2.

Sumber yang digunakan:
  [1] Pakan Fermentasi (Silase) - "Silase adalah pakan hijauan yang diawetkan melalui proses fermentasi anaerob oleh bakteri asam laktat. Bahan yang umum d..."
  [2] Pakan Fermentasi (Silase) - "dapat mempercepat fermentasi. Keunggulan silase: dapat disimpan 6-12 bulan, mengurangi kehilangan nutrien dibanding hija..."
  [3] Sapi Potong dan Penggemukan - "akhir 450-550 kg. Rasio pakan konsentrat:hijauan untuk penggemukan adalah 70:30 hingga 80:20. Karkas yang baik memiliki ..."
----------------------------------------------------------------------


{'result': 'Proses fermentasi silase berlangsung selama 21 hari, dan pH yang ideal adalah 3.8-4.2.',
 'source_documents': [Document(id='91215da6-ecc1-49a8-be81-e06b45c3184d', metadata={'title': 'Pakan Fermentasi (Silase)'}, page_content='Silase adalah pakan hijauan yang diawetkan melalui proses fermentasi anaerob\noleh bakteri asam laktat. Bahan yang umum digunakan adalah rumput gajah, jagung, atau\nsorgum dengan kadar air 60-70%. Proses fermentasi berlangsung 21 hari dalam kondisi\nanaerob (tanpa oksigen). pH silase yang baik adalah 3.8-4.2. Penambahan molases 3-5%'),
  Document(id='6cf193ac-02e4-4b4e-9e24-3d57ebe6726b', metadata={'title': 'Pakan Fermentasi (Silase)'}, page_content='dapat mempercepat fermentasi. Keunggulan silase: dapat disimpan 6-12 bulan, mengurangi\nkehilangan nutrien dibanding hijauan kering, dan tersedia sepanjang tahun terutama di\nmusim kemarau.'),
  Document(id='fcc45d78-4ff1-4e50-9c71-62f29205ccc4', metadata={'title': 'Sapi Potong dan Penggemukan'}, page_cont

### 5.3 Analisis Hasil Evaluasi

#### Kelebihan Sistem
1. **Akurasi tinggi untuk pertanyaan faktual** - Sistem berhasil menjawab pertanyaan spesifik seperti masa kebuntingan sapi (280-285 hari) dan kebutuhan protein kasar sapi perah (16-18%) dengan angka yang persis sesuai knowledge base.
2. **Tahan terhadap halusinasi** - Untuk pertanyaan di luar topik (misalnya resep brownies), sistem menolak menjawab sesuai instruksi prompt.
3. **Multilingual retrieval bekerja baik** - Embedding multilingual berhasil menangkap kemiripan semantik dalam bahasa Indonesia.
4. **Knowledge base mudah diperbarui** - Fungsi `add_document()` dan `replace_knowledge_base()` memungkinkan ekspansi KB tanpa perlu mengubah kode utama.
5. **Source attribution** - Setiap jawaban menyertakan sumber dokumen, sehingga pengguna dapat memverifikasi.

#### Keterbatasan dan Kekurangan
1. **Chunk size tetap (400)** - Tidak optimal untuk dokumen pendek; potongan kalimat di tengah konteks bisa hilang. Chunking semantik berbasis paragraf akan lebih baik.
2. **Top-k retrieval terbatas (k=3)** - Untuk pertanyaan kompleks yang butuh sintesis dari banyak dokumen, k=3 mungkin tidak cukup.
3. **Tidak ada re-ranking** - Hasil retrieval murni berdasarkan cosine similarity, tanpa re-ranker untuk meningkatkan relevansi.
4. **Tidak ada memory percakapan** - Setiap pertanyaan diperlakukan independen; tidak ada follow-up question yang merujuk konteks sebelumnya.
5. **Ketergantungan pada Groq API** - Jika kuota habis atau API down, sistem tidak bisa menjawab.
6. **Knowledge base masih terbatas** - Hanya sekitar 11 dokumen, belum mencakup seluruh sub-bidang Animal Science (misalnya teknologi reproduksi seperti embryo transfer, akuakultur, dll).
7. **Tidak ada evaluasi otomatis** - Belum menggunakan metric seperti RAGAS (faithfulness, answer relevancy, context precision) untuk evaluasi kuantitatif.


## 6. Insight dan Recommendation Action

### Insight yang Diperoleh

1. **Kualitas knowledge base sama dengan kualitas jawaban.** Sistem RAG hanya sebaik knowledge base-nya. Dokumen yang ditulis terstruktur (judul + isi padat fakta) menghasilkan retrieval yang jauh lebih akurat dibanding teks panjang naratif.

2. **Bahasa dan domain matter.** Embedding monolingual English akan gagal pada KB berbahasa Indonesia. Memilih embedding multilingual adalah keputusan kritis untuk konteks Indonesian Animal Science.

3. **Trade-off chunk size**: chunk terlalu kecil akan kehilangan konteks; chunk terlalu besar akan menghasilkan noise di dalam retrieval. Untuk domain teknis seperti Animal Science yang penuh angka spesifik, chunk sekitar 300-500 karakter ideal.

4. **Prompt engineering sangat penting.** Tanpa instruksi eksplisit "jangan mengarang", LLM cenderung tetap menjawab pertanyaan di luar KB dengan informasi dari pre-training-nya.

5. **Update KB harus diikuti rebuild vector store.** Lupa rebuild sama dengan jawaban berdasarkan KB lama, sebuah bug halus yang sulit di-debug.

### Rekomendasi Pengembangan Lanjutan

1. **Perluas knowledge base** - Tambahkan dokumen dari jurnal peternakan, FAO reports, dan textbook Animal Science. Target minimal 100+ dokumen yang mencakup semua sub-bidang.

2. **Implementasi hybrid search** - Kombinasikan dense retrieval (embedding) dengan sparse retrieval (BM25) untuk menangani query yang berisi istilah teknis spesifik (misal nama bangsa sapi, kode vaksin).

3. **Tambahkan re-ranking** - Gunakan cross-encoder seperti `ms-marco-MiniLM-L-6-v2` untuk me-rank ulang hasil retrieval sebelum di-feed ke LLM.

4. **Conversational memory** - Tambahkan `ConversationBufferMemory` agar chatbot bisa menangani follow-up question seperti "Lalu bagaimana dengan domba?" setelah membahas kambing.

5. **Evaluasi otomatis dengan RAGAS** - Implementasikan metrik faithfulness, answer relevancy, context precision, dan context recall untuk evaluasi kuantitatif.

6. **UI interaktif** - Bungkus dengan Gradio atau Streamlit untuk demo yang user-friendly.

7. **Multi-modal RAG** - Tambahkan kemampuan retrieval dari gambar (misalnya foto gejala penyakit) menggunakan CLIP embedding.

8. **Metadata filtering** - Manfaatkan metadata (kategori: Nutrisi, Reproduksi, Kesehatan) untuk memfilter retrieval berdasarkan topik pertanyaan.

9. **Caching jawaban** - Untuk pertanyaan yang sering muncul, simpan jawaban di cache (Redis atau SQLite) untuk mengurangi latensi dan biaya API.

10. **Guardrails untuk advice medis** - Untuk pertanyaan kesehatan hewan, tambahkan disclaimer agar pengguna tetap berkonsultasi dengan dokter hewan profesional.


#### 7. Reflection Question

#### Pertanyaan 1
> Setelah membangun aplikasi RAG, bagian mana dari proses pengambilan informasi (retrieval) yang membuatmu paling memahami pentingnya kualitas knowledge base dalam menghasilkan jawaban yang relevan? Jelaskan bagaimana pengalaman ini mengubah caramu melihat proses pencarian informasi dalam sebuah chatbot.

**Jawaban:**

Bagian yang paling membuka pemahaman saya adalah proses chunking dan similarity search. Saat saya mencoba mengajukan pertanyaan spesifik seperti "Berapa kebutuhan protein kasar sapi perah?", sistem dapat menjawab dengan akurat (16-18%) hanya karena dokumen di knowledge base menulis fakta tersebut secara eksplisit dan terstruktur. Sebaliknya, ketika saya menambahkan dokumen yang lebih naratif atau panjang, retrieval menjadi kurang presisi karena chunk yang diambil sering hanya berisi sebagian informasi.

Pengalaman ini mengubah cara saya melihat chatbot:

- Sebelumnya, saya pikir chatbot pintar karena LLM-nya canggih. Sekarang, saya sadar bahwa chatbot RAG sebenarnya 70% bergantung pada kualitas knowledge base dan retrieval, dan hanya 30% pada LLM. LLM secanggih apapun tidak bisa menjawab dengan benar jika dokumen yang ter-retrieve tidak mengandung jawaban.
- "Garbage in, garbage out" menjadi nyata: dokumen dengan informasi rancu, redundant, atau tidak terstruktur akan menghasilkan jawaban yang membingungkan.
- Saya juga menyadari bahwa cara dokumen ditulis (format, struktur, panjang paragraf) sangat berpengaruh pada hasil retrieval, bahkan lebih besar pengaruhnya dibanding pemilihan model embedding.
- Pencarian informasi dalam chatbot bukan sekadar Google semantic, tapi sebuah pipeline rumit di mana setiap tahap (chunking, embedding, indexing, similarity threshold, prompt construction) dapat menjadi titik kegagalan.

Kesimpulannya: membangun chatbot yang bagus sama dengan membangun knowledge base yang bagus terlebih dahulu, baru memikirkan komponen teknis lainnya.

---

#### Pertanyaan 2
> Dalam memilih LLM, embedding model, dan vector database, apa hal yang membuatmu menyadari bahwa arsitektur teknis sebuah sistem RAG harus disesuaikan dengan kebutuhan pengguna dan konteks data? Jelaskan bagaimana keputusan teknismu dipengaruhi oleh karakteristik topik knowledge base yang kamu gunakan.

**Jawaban:**

Kesadaran terbesar muncul ketika saya menyadari bahwa setiap pilihan teknis adalah trade-off yang dipengaruhi oleh karakteristik data dan kebutuhan pengguna:

**1. Pemilihan Embedding Model - dipengaruhi karakteristik bahasa:**
- Knowledge base saya berbahasa Indonesia. Jika saya memakai embedding default seperti `all-MiniLM-L6-v2` (English-only), retrieval akan kacau karena model tidak memahami semantik bahasa Indonesia.
- Saya akhirnya memilih `paraphrase-multilingual-MiniLM-L12-v2` karena mendukung 50+ bahasa termasuk Indonesia. Ini menunjukkan bahwa konteks bahasa data harus menjadi pertimbangan pertama dalam memilih embedding.

**2. Pemilihan LLM - dipengaruhi sifat domain dan budget:**
- Topik Animal Science penuh dengan fakta spesifik dan angka (durasi kebuntingan, persentase nutrien, suhu kandang, dll). Saya butuh LLM dengan instruction following yang kuat agar mau membaca konteks dengan teliti dan tidak mengarang.
- Karena ini proyek eksperimen, budget juga jadi pertimbangan. Memilih Llama 3.3 70B via Groq (free tier) memberikan keseimbangan: kualitas reasoning bagus, biaya nol, dan latency rendah.
- Jika topik adalah hal yang sensitif (misalnya diagnosis penyakit hewan kritis), saya akan memilih LLM yang lebih konservatif atau menambahkan layer validasi.

**3. Pemilihan Vector DB - dipengaruhi skala data dan use case:**
- Knowledge base saya kecil (sekitar 11 dokumen, puluhan chunk). Tidak perlu solusi enterprise seperti Pinecone atau Weaviate yang punya overhead infrastruktur.
- ChromaDB local sudah cukup, bisa di-persist ke disk, dan tidak butuh server eksternal. Cocok untuk demo di Colab.
- Jika nantinya KB scaling ke jutaan dokumen, saya akan pertimbangkan migrasi ke vector DB yang dioptimasi untuk skala besar dengan HNSW indexing dan sharding.

**4. Chunk Size - dipengaruhi struktur dokumen:**
- Dokumen Animal Science saya berbentuk paragraf padat fakta (sekitar 600-800 karakter). Chunk size 400 dengan overlap 50 cocok karena menjaga konteks tetap utuh per topik tanpa kehilangan keterkaitan antar kalimat.

**Insight kunci:**
Tidak ada "stack RAG terbaik" yang universal. Sebuah RAG untuk dokumen hukum berbahasa Inggris akan butuh stack yang sangat berbeda dengan RAG untuk Animal Science berbahasa Indonesia. Arsitektur harus dirancang dari kebutuhan pengguna dan karakteristik data, bukan dari teknologi yang sedang tren. Memilih komponen tanpa memahami konteks justru bisa menghasilkan sistem yang mahal tapi performanya buruk.

---
